In [18]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
  accuracy_score, 
  precision_score, 
  recall_score, 
  f1_score,
  roc_auc_score, 
  confusion_matrix,
  RocCurveDisplay,
  PrecisionRecallDisplay
) 


PROJECT_ROOT = Path.cwd().parents[0]  # adjust if needed
sys.path.append(str(PROJECT_ROOT))

In [10]:

RANDOM_STATE = 42
TEST_SIZE = 0.2
THRESHOLD = 0.5

# Path to the SAME dataset MAS used for evaluation
DATASET_PATH = "../data/eval/diabetes_eval_merged.csv"  # adjust if needed

# Split cache (so MAS + baseline can literally reuse the exact same indices)
SPLIT_CACHE_PATH = "artifacts/baseline_split_indices_v1.npz"
os.makedirs(os.path.dirname(SPLIT_CACHE_PATH), exist_ok=True)

In [7]:
try:
    from agents.data_agent import FEATURES, TARGET, build_preprocessor
except ModuleNotFoundError:
    from data_agent import FEATURES, TARGET, build_preprocessor

FEATURES, TARGET

(['gender',
  'age',
  'bmi',
  'glucose',
  'hba1c',
  'blood_pressure',
  'hypertension',
  'heart_disease',
  'smoking_history',
  'insulin'],
 'diabetes_present')

In [11]:
df = pd.read_csv(DATASET_PATH)

# Keep only what the MAS pipeline expects
needed = [c for c in FEATURES if c in df.columns] + ([TARGET] if TARGET in df.columns else [])
df = df[needed].copy()

# Match train_model.py behavior: drop rows missing target
df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = df[TARGET].astype(int)

X = df[FEATURES].copy()
y = df[TARGET].copy()

df.shape, y.value_counts().to_dict()


((96914, 11), {0: 88164, 1: 8750})

In [12]:
if os.path.exists(SPLIT_CACHE_PATH):
    cache = np.load(SPLIT_CACHE_PATH, allow_pickle=True)
    train_idx = cache["train_idx"]
    test_idx  = cache["test_idx"]
else:
    # If your MAS split did NOT use stratify, set stratify=None here to match.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,  # <- set to None if MAS didn't stratify
    )
    train_idx = X_train.index.to_numpy()
    test_idx  = X_test.index.to_numpy()
    np.savez(SPLIT_CACHE_PATH, train_idx=train_idx, test_idx=test_idx)

X_train = X.loc[train_idx].copy()
y_train = y.loc[train_idx].copy()
X_test  = X.loc[test_idx].copy()
y_test  = y.loc[test_idx].copy()

(X_train.shape, X_test.shape, y_train.mean(), y_test.mean())


((77531, 10),
 (19383, 10),
 np.float64(0.0902864660587378),
 np.float64(0.09028530155290719))

In [13]:
preprocessor = build_preprocessor()

X_train_p = preprocessor.fit_transform(X_train)
X_test_p  = preprocessor.transform(X_test)

X_train_p.shape, X_test_p.shape


((77531, 19), (19383, 19))

In [14]:
baseline = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
baseline.fit(X_train_p, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [15]:
proba = baseline.predict_proba(X_test_p)[:, 1]
pred  = (proba >= THRESHOLD).astype(int)

acc  = accuracy_score(y_test, pred)
prec = precision_score(y_test, pred, zero_division=0)
rec  = recall_score(y_test, pred, zero_division=0)
f1   = f1_score(y_test, pred, zero_division=0)

# ROC-AUC needs probabilities
auc = roc_auc_score(y_test, proba)

cm = confusion_matrix(y_test, pred)

print("=== Baseline Evaluation (binary) ===")
print(f"N: {len(y_test)}")
print(f"Threshold: {THRESHOLD}")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1:        {f1:.4f}")
print(f"ROC-AUC:   {auc:.4f}")
print(f"Confusion: {cm.tolist()}")  # [[TN, FP],[FN, TP]]
print("====================================")


=== Baseline Evaluation (binary) ===
N: 19383
Threshold: 0.5
Accuracy:  0.9569
Precision: 0.8622
Recall:    0.6223
F1:        0.7229
ROC-AUC:   0.9513
Confusion: [[17459, 174], [661, 1089]]


In [16]:
baseline_metrics = {
    "N": int(len(y_test)),
    "threshold": float(THRESHOLD),
    "accuracy": float(acc),
    "precision": float(prec),
    "recall": float(rec),
    "f1": float(f1),
    "roc_auc": float(auc),
    "confusion_matrix": cm.tolist(),
}
baseline_metrics


{'N': 19383,
 'threshold': 0.5,
 'accuracy': 0.9569210132590414,
 'precision': 0.8622327790973872,
 'recall': 0.6222857142857143,
 'f1': 0.7228675738466644,
 'roc_auc': 0.9512728082896518,
 'confusion_matrix': [[17459, 174], [661, 1089]]}